# LeetCode #1219: Path with Maximum Gold

https://leetcode.com/problems/path-with-maximum-gold/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(m \times n \times 4^{m \times n})$ | $O(m \times n)$ |
| **Optimal: DFS Backtracking ★** | $O(m \times n \times 4^k)$ | $O(m \times n)$ |

---

## Understanding the Methods

### Brute Force
Try every possible path from every cell exhaustively without pruning zero-gold cells. This visits unreachable states and is far slower in practice.

### Optimal: DFS Backtracking ★
Start DFS from every non-zero cell, temporarily marking visited cells as 0 to prevent revisiting, then restoring the original value after the recursive call. $k$ is the number of reachable non-zero cells, which is typically much smaller than $m \times n$, making this very fast in practice despite the worst-case exponent.

**Constraints:**
* $1 \leq m, n \leq 15$
* $0 \leq \text{grid}[i][j] \leq 100$
* There are at most 25 non-zero cells.

## Solutions

### C#

In [ ]:
public class Solution {
    private int[][] _grid;
    private int _rows, _cols, _best;
    private int[] _dr = {0,0,1,-1};
    private int[] _dc = {1,-1,0,0};

    public int GetMaximumGold(int[][] grid) {
        _grid = grid; _rows = grid.Length; _cols = grid[0].Length;
        for (int r = 0; r < _rows; r++)
            for (int c = 0; c < _cols; c++)
                if (grid[r][c] != 0) Dfs(r, c, 0);
        return _best;
    }

    private void Dfs(int r, int c, int collected) {
        // Collect gold at current cell before exploring neighbors
        collected += _grid[r][c];
        _best = Math.Max(_best, collected);
        int saved = _grid[r][c];
        // Mark visited to block re-entry on this path
        _grid[r][c] = 0;
        for (int d = 0; d < 4; d++) {
            int nr = r + _dr[d], nc = c + _dc[d];
            if (nr >= 0 && nr < _rows && nc >= 0 && nc < _cols && _grid[nr][nc] != 0)
                Dfs(nr, nc, collected);
        }
        // Restore so other starting paths see the original value
        _grid[r][c] = saved;
    }
}

### Python

In [ ]:
class Solution:
    def get_maximum_gold(self, grid: list[list[int]]) -> int:
        rows, cols = len(grid), len(grid[0])
        best = 0

        def dfs(r: int, c: int, collected: int) -> None:
            nonlocal best
            # Collect gold at current cell before exploring neighbors
            collected += grid[r][c]
            best = max(best, collected)
            saved = grid[r][c]
            # Mark visited to block re-entry on this path
            grid[r][c] = 0
            for dr, dc in ((0,1),(0,-1),(1,0),(-1,0)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != 0:
                    dfs(nr, nc, collected)
            # Restore so other starting paths see the original value
            grid[r][c] = saved

        for r in range(rows):
            for c in range(cols):
                if grid[r][c] != 0:
                    dfs(r, c, 0)
        return best

### Go

In [ ]:
func getMaximumGold(grid [][]int) int {
    rows, cols, best := len(grid), len(grid[0]), 0
    dirs := [][2]int{{0,1},{0,-1},{1,0},{-1,0}}

    var dfs func(r, c, collected int)
    dfs = func(r, c, collected int) {
        // Collect gold at current cell before exploring neighbors
        collected += grid[r][c]
        if collected > best { best = collected }
        saved := grid[r][c]
        // Mark visited to block re-entry on this path
        grid[r][c] = 0
        for _, d := range dirs {
            nr, nc := r+d[0], c+d[1]
            if nr >= 0 && nr < rows && nc >= 0 && nc < cols && grid[nr][nc] != 0 {
                dfs(nr, nc, collected)
            }
        }
        // Restore so other starting paths see the original value
        grid[r][c] = saved
    }

    for r := 0; r < rows; r++ {
        for c := 0; c < cols; c++ {
            if grid[r][c] != 0 { dfs(r, c, 0) }
        }
    }
    return best
}

### Rust

In [ ]:
impl Solution {
    pub fn get_maximum_gold(mut grid: Vec<Vec<i32>>) -> i32 {
        let (rows, cols) = (grid.len(), grid[0].len());
        let mut best = 0i32;
        for r in 0..rows {
            for c in 0..cols {
                if grid[r][c] != 0 {
                    Self::dfs(&mut grid, r, c, 0, rows, cols, &mut best);
                }
            }
        }
        best
    }

    fn dfs(grid: &mut Vec<Vec<i32>>, r: usize, c: usize, collected: i32, rows: usize, cols: usize, best: &mut i32) {
        // Collect gold at current cell before exploring neighbors
        let collected = collected + grid[r][c];
        *best = (*best).max(collected);
        let saved = grid[r][c];
        // Mark visited to block re-entry on this path
        grid[r][c] = 0;
        let dirs: [(i32,i32); 4] = [(0,1),(0,-1),(1,0),(-1,0)];
        for (dr, dc) in dirs {
            let (nr, nc) = (r as i32 + dr, c as i32 + dc);
            if nr >= 0 && nr < rows as i32 && nc >= 0 && nc < cols as i32 {
                let (nr, nc) = (nr as usize, nc as usize);
                if grid[nr][nc] != 0 {
                    Self::dfs(grid, nr, nc, collected, rows, cols, best);
                }
            }
        }
        // Restore so other starting paths see the original value
        grid[r][c] = saved;
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `grid = [[0,6,0],[5,8,7],[0,9,0]]`
Starting from $(1,1)$ (value $8$) and moving to $(2,1)$ (value $9$) yields $8+9=17$, but starting from $(2,1)$ then up to $(1,1)$ and right to $(1,2)$ gives $9+8+7=24$. The DFS explores all branches and records the maximum.

### 2. Slightly Complex
**Input:** `grid = [[1,0,7],[2,0,6],[3,4,5],[0,3,0],[9,0,20]]`
Two disconnected gold regions; the DFS from each non-zero cell explores its connected component independently. The bottom-right region containing $20$ dominates.

### 3. Edge Case: Time Factor
**Input:** `grid` with all 25 non-zero cells forming a fully connected snake (no zeros)
Every cell can reach every other cell in some order, giving the DFS up to $4^{25}$ paths to explore. The constraint of at most 25 non-zero cells is precisely the bound that keeps this tractable.

### 4. Edge Case: Space Factor
**Input:** `grid = [[100,0,0,...,0], ...]` — single non-zero cell in a $15 \times 15$ grid
DFS starts and terminates immediately — only one cell is reachable. The call stack depth is $O(1)$ and the answer is $100$. Marking-and-restoring has no observable effect.

### 5. Almost-Impossible but Plausible
**Input:** `grid` with 25 cells each valued $100$, arranged in a line
Maximum possible gold is $25 \times 100 = 2500$. The backtracker must confirm that the unique linear path collects all cells without revisiting — verifying the mark-and-restore mechanism correctly prevents self-intersection even in a dense layout.